# XQuality Checkpoint-to-Output LLM Ablation — Exact Evaluation Only + Native NeoOLAF Reference

This notebook **does not call OpenRouter**. It reuses the outputs generated by the v5 batched JSONL notebook and evaluates them with the same NeoOLAF extraction evaluation stack used by the original layer ablation (`evaluate_extraction`, `get_profile`, `EvaluationArtifact`).

Important: the checkpoint-to-output curve evaluates **LLM-finalizer outputs generated from each checkpoint**, not the native NeoOLAF pipeline output. Therefore the point named `layer12_serialization` is **not** the native NeoOLAF final export. It is the result of asking an LLM to regenerate triples from the Layer 12 checkpoint.

This version also adds a **native NeoOLAF final-export reference** to the table/plot, so the LLM-finalizer ablation is not confused with the real final NeoOLAF output.


In [ ]:

from __future__ import annotations

import json
import re
import sys
import time
from pathlib import Path
from typing import Any
from collections import Counter

import pandas as pd
from tqdm.auto import tqdm

# Locate NeoOLAF repository root.
def find_repo_root(start: Path | None = None) -> Path:
    start = Path(start or Path.cwd()).resolve()
    candidates = [start] + list(start.parents)
    for p in candidates:
        if (p / "src" / "neoolaf").exists() and (p / "examples").exists():
            return p
    raise RuntimeError(f"Could not find NeoOLAF repo root from {start}")

ROOT = find_repo_root()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print("ROOT:", ROOT)
print("SRC:", SRC)


In [ ]:

# ---------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------
PROFILE_NAME = "xquality_loose"  # same default as layer_ablation_machine32.ipynb; set to "xquality_relaxed_recall" to match the final-export gold notebook

RUNS_ROOT = ROOT / "examples" / "XQualityMachine32" / "runs"

# This is the output folder produced by the v5 batched JSONL notebook.
# If your v5 notebook used a different folder name, change this line only.
V5_OUTPUT_ROOT = RUNS_ROOT / "checkpoint_to_output_llm_ablation_batched_jsonl_robust"

# This notebook writes exact-evaluation results here.
EVAL_OUTPUT_ROOT = RUNS_ROOT / "checkpoint_to_output_llm_ablation_v5_exact_eval_only"
EVAL_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

GOLD_JSON = ROOT / "data" / "XQuality" / "Examples" / "XQuality_all_triplets_flat_en.json"
GOLD_EXCEL = ROOT / "data" / "XQuality" / "Machine32" / "machine32_triples.xlsx"
GOLD_PATH = GOLD_JSON if GOLD_JSON.exists() else GOLD_EXCEL

# If True, overwrite exact_eval.json files even when they already exist.
FORCE_REEVALUATE = True


# Native NeoOLAF final export reference.
# This is the REAL final NeoOLAF output, unlike the LLM-finalizer layer12 point.
NATIVE_EXPORT_DIR = RUNS_ROOT / "xquality_machine32" / "layer12_from_l11" / "exports"
NATIVE_EXPORT_EVAL_DIR = RUNS_ROOT / "xquality_machine32" / "evaluations" / "gold_layer12_native_reference_from_v5_eval"
SEED_ONTOLOGY = ROOT / "data" / "ontology" / "ContextOntology-COInd4.owl"

# If True, this notebook will run the native export evaluator if no metrics.summary.json is found.
# This does not call OpenRouter; it only runs NeoOLAF evaluation locally.
RUN_NATIVE_EXPORT_EVAL_IF_MISSING = True

# Use the same profile for native and LLM-finalizer by default.
# If you want to reproduce the 0.6107 relation F1 you quoted from the no-gold notebook,
# set this to "xquality_relaxed_recall" and also set PROFILE_NAME accordingly for apples-to-apples comparison.
NATIVE_EXPORT_PROFILE = PROFILE_NAME

print("V5_OUTPUT_ROOT:", V5_OUTPUT_ROOT, "exists=", V5_OUTPUT_ROOT.exists())
print("EVAL_OUTPUT_ROOT:", EVAL_OUTPUT_ROOT)
print("GOLD_JSON:", GOLD_JSON, "exists=", GOLD_JSON.exists())
print("GOLD_EXCEL:", GOLD_EXCEL, "exists=", GOLD_EXCEL.exists())
print("GOLD_PATH:", GOLD_PATH, "exists=", GOLD_PATH.exists())
assert V5_OUTPUT_ROOT.exists(), f"V5 output folder not found: {V5_OUTPUT_ROOT}"
assert GOLD_PATH.exists(), f"Gold file not found: {GOLD_PATH}"

print("NATIVE_EXPORT_DIR:", NATIVE_EXPORT_DIR, "exists=", NATIVE_EXPORT_DIR.exists())
print("NATIVE_EXPORT_PROFILE:", NATIVE_EXPORT_PROFILE)


In [ ]:

# ---------------------------------------------------------------------
# Import the exact NeoOLAF evaluation stack
# ---------------------------------------------------------------------
from neoolaf.evaluation.runners.evaluate_layer_state import load_xquality_gold_any, gold_to_artifact
from neoolaf.evaluation.metrics.extraction import evaluate_extraction
from neoolaf.evaluation.profiles.registry import get_profile
from neoolaf.evaluation.schema.artifact import (
    EvaluationArtifact,
    EvalDocument,
    EvalEntity,
    EvalRelation,
)

profile = get_profile(PROFILE_NAME)
gold = load_xquality_gold_any(GOLD_PATH)
gold_artifact = gold_to_artifact(gold, profile=PROFILE_NAME)

print("Loaded gold with exact NeoOLAF loader.")
print("Profile:", PROFILE_NAME)
print("Gold entities:", sum(len(v) for v in gold_artifact.entities_by_doc.values()))
print("Gold relations:", sum(len(v) for v in gold_artifact.relations_by_doc.values()))


In [ ]:

# ---------------------------------------------------------------------
# Find v5 layer outputs
# ---------------------------------------------------------------------

def find_v5_layer_outputs(root: Path) -> pd.DataFrame:
    rows = []
    for d in sorted(root.iterdir()):
        if not d.is_dir():
            continue
        triples_path = d / "llm_triples.json"
        if not triples_path.exists():
            # tolerate alternative names
            alt = list(d.glob("*triples*.json"))
            triples_path = alt[0] if alt else triples_path
        if triples_path.exists():
            meta_path = d / "batch_metadata.csv"
            metrics_path = d / "metrics.json"
            rows.append({
                "layer_name": d.name,
                "layer_dir": str(d),
                "triples_path": str(triples_path),
                "batch_metadata_path": str(meta_path) if meta_path.exists() else "",
                "old_metrics_path": str(metrics_path) if metrics_path.exists() else "",
            })
    return pd.DataFrame(rows)

layers_df = find_v5_layer_outputs(V5_OUTPUT_ROOT)
print("Found layer outputs:", len(layers_df))
display(layers_df)
assert len(layers_df) > 0, f"No llm_triples.json files found under {V5_OUTPUT_ROOT}"


In [ ]:

# ---------------------------------------------------------------------
# Convert LLM-finalizer triples to NeoOLAF EvaluationArtifact
# ---------------------------------------------------------------------

def clean_label(x: Any) -> str:
    if x is None:
        return ""
    s = str(x).strip()
    if not s or s.lower() in {"nan", "none", "null"}:
        return ""
    return re.sub(r"\s+", " ", s)


def canonical_predicate(x: Any) -> str:
    s = clean_label(x).upper()
    s = re.sub(r"[^A-Z0-9]+", "_", s).strip("_")
    aliases = {
        "TRIGGER": "TRIGGERS",
        "TRIGGERS": "TRIGGERS",
        "CAUSE": "CAUSES",
        "CAUSES": "CAUSES",
        "EFFECT": "CAUSES",
        "EFFET": "CAUSES",
        "REQUIRES": "REQUIRES",
        "REQUIRE": "REQUIRES",
        "INTERVENTION": "REQUIRES",
        "ACTION": "REQUIRES",
        "HANDLED_BY": "HANDLED_BY",
        "RESPONSIBLE": "HANDLED_BY",
        "RESPONSABLE": "HANDLED_BY",
        "REFERENCES": "REFERENCES",
        "REFERENCE": "REFERENCES",
        "REF": "REFERENCES",
    }
    return aliases.get(s, s)


def load_llm_triples(path: str | Path) -> list[dict[str, Any]]:
    path = Path(path)
    data = json.loads(path.read_text(encoding="utf-8"))
    if isinstance(data, dict):
        if "triples" in data:
            data = data["triples"]
        elif "candidate_triples" in data:
            data = data["candidate_triples"]
        else:
            data = []
    if not isinstance(data, list):
        return []

    out = []
    seen = set()
    for i, t in enumerate(data):
        if not isinstance(t, dict):
            continue
        subj = clean_label(t.get("subject_label") or t.get("subject") or t.get("head") or t.get("s"))
        pred = canonical_predicate(t.get("predicate") or t.get("predicate_label") or t.get("relation") or t.get("p"))
        obj = clean_label(t.get("object_label") or t.get("object") or t.get("tail") or t.get("o"))
        if not subj or not pred or not obj:
            continue
        key = (subj.lower(), pred, obj.lower())
        if key in seen:
            continue
        seen.add(key)
        out.append({
            "triple_id": t.get("triple_id") or f"llm_{len(out):05d}",
            "subject_label": subj,
            "predicate": pred,
            "object_label": obj,
            "confidence": t.get("confidence"),
            "source": t,
        })
    return out


def artifact_from_llm_triples(
    triples: list[dict[str, Any]],
    *,
    layer_name: str,
    run_id: str,
    profile_name: str = PROFILE_NAME,
) -> EvaluationArtifact:
    doc_id = "xquality"
    artifact = EvaluationArtifact(
        method="llm_finalizer",
        dataset="xquality",
        profile=profile_name,
        run_id=run_id,
        metadata={
            "layer_name": layer_name,
            "source_state": "LLM checkpoint-to-output finalizer",
            "evaluation_mode": "exact_neoolaf_evaluate_extraction",
        },
    )
    artifact.documents.append(EvalDocument(document_id=doc_id, metadata={"dataset": "xquality"}))

    entities = {}
    relations = []

    def add_entity(label: str, type_: str | None = None, raw: Any | None = None):
        label = clean_label(label)
        if not label:
            return
        key = label.lower()
        if key not in entities:
            entities[key] = EvalEntity(
                label=label,
                type=type_,
                provenance_present=False,
                raw=raw or {},
            )

    for t in triples:
        h = clean_label(t.get("subject_label"))
        r = canonical_predicate(t.get("predicate"))
        o = clean_label(t.get("object_label"))
        if not h or not r or not o:
            continue
        add_entity(h, type_="llm_relation_source", raw=t.get("source", {}))
        add_entity(o, type_="llm_relation_target", raw=t.get("source", {}))
        relations.append(EvalRelation(
            head=h,
            relation=r,
            tail=o,
            confidence=t.get("confidence"),
            provenance_present=False,
            raw=t.get("source", t),
        ))

    artifact.entities_by_doc[doc_id] = sorted(entities.values(), key=lambda e: e.label.lower())
    artifact.relations_by_doc[doc_id] = relations
    return artifact


In [ ]:

# ---------------------------------------------------------------------
# Exact NeoOLAF evaluation utilities
# ---------------------------------------------------------------------

def flatten_scalars(obj: Any, prefix: str = "") -> dict[str, Any]:
    out = {}
    if isinstance(obj, dict):
        for k, v in obj.items():
            key = f"{prefix}.{k}" if prefix else str(k)
            out.update(flatten_scalars(v, key))
    elif isinstance(obj, (list, tuple)):
        # Do not explode long lists into summary; keep their length.
        out[prefix + ".len" if prefix else "len"] = len(obj)
    else:
        if isinstance(obj, (str, int, float, bool)) or obj is None:
            out[prefix] = obj
    return out


def pick_metric(flat: dict[str, Any], candidates: list[str]):
    # Exact candidates first.
    for c in candidates:
        if c in flat:
            return flat[c]
    # Suffix candidates.
    for c in candidates:
        for k, v in flat.items():
            if k.endswith(c):
                return v
    return None


def summarize_extraction(extraction: dict[str, Any]) -> dict[str, Any]:
    flat = flatten_scalars(extraction)
    row = {}
    row.update(flat)

    # Common aliases for easier plotting/reading. These remain robust if exact key names vary.
    row["entity_precision"] = pick_metric(flat, [
        "entity.precision", "entities.precision", "entity_metrics.precision", "entity.p", "entity_p", "entities.p"
    ])
    row["entity_recall"] = pick_metric(flat, [
        "entity.recall", "entities.recall", "entity_metrics.recall", "entity.r", "entity_r", "entities.r"
    ])
    row["entity_f1"] = pick_metric(flat, [
        "entity.f1", "entities.f1", "entity_metrics.f1", "entity_f1", "entities.f1_score"
    ])
    row["relation_precision"] = pick_metric(flat, [
        "relation.precision", "relations.precision", "relation_metrics.precision", "relation.p", "relation_p", "relations.p"
    ])
    row["relation_recall"] = pick_metric(flat, [
        "relation.recall", "relations.recall", "relation_metrics.recall", "relation.r", "relation_r", "relations.r"
    ])
    row["relation_f1"] = pick_metric(flat, [
        "relation.f1", "relations.f1", "relation_metrics.f1", "relation_f1", "relations.f1_score"
    ])
    return row


def extract_per_relation_rows(extraction: dict[str, Any], *, layer_name: str) -> list[dict[str, Any]]:
    rows = []
    # Try common locations.
    candidates = []
    for key in ["per_relation", "per_relation_metrics", "relations_by_type", "by_relation"]:
        if isinstance(extraction, dict) and isinstance(extraction.get(key), dict):
            candidates.append(extraction[key])
    # Recursive fallback: find dicts containing relation names.
    def rec(x):
        if isinstance(x, dict):
            keys = set(str(k).upper() for k in x.keys())
            if keys & {"TRIGGERS", "CAUSES", "REQUIRES", "HANDLED_BY", "REFERENCES"}:
                candidates.append(x)
            for v in x.values():
                rec(v)
        elif isinstance(x, list):
            for v in x:
                rec(v)
    rec(extraction)

    seen = set()
    for d in candidates:
        for rel, metrics in d.items():
            rel_up = str(rel).upper()
            if rel_up not in {"TRIGGERS", "CAUSES", "REQUIRES", "HANDLED_BY", "REFERENCES"}:
                continue
            key = (layer_name, rel_up)
            if key in seen:
                continue
            seen.add(key)
            row = {"layer_name": layer_name, "predicate": rel_up}
            if isinstance(metrics, dict):
                row.update(flatten_scalars(metrics))
            else:
                row["value"] = metrics
            rows.append(row)
    return rows


In [ ]:

# ---------------------------------------------------------------------
# Evaluate all V5 outputs using exact NeoOLAF evaluator
# ---------------------------------------------------------------------
summary_rows = []
per_relation_rows = []

for _, row in tqdm(layers_df.iterrows(), total=len(layers_df), desc="Exact evaluation of V5 outputs"):
    layer_name = row["layer_name"]
    triples_path = Path(row["triples_path"])
    layer_eval_dir = EVAL_OUTPUT_ROOT / layer_name
    layer_eval_dir.mkdir(parents=True, exist_ok=True)
    output_json = layer_eval_dir / "exact_eval.json"
    output_artifact = layer_eval_dir / "llm_eval_artifact.json"

    if output_json.exists() and not FORCE_REEVALUATE:
        result = json.loads(output_json.read_text(encoding="utf-8"))
        extraction = result.get("extraction", {})
        triples_count = result.get("counts", {}).get("pred_triples")
    else:
        triples = load_llm_triples(triples_path)
        pred_artifact = artifact_from_llm_triples(
            triples,
            layer_name=layer_name,
            run_id=f"llm_finalizer_{layer_name}",
            profile_name=PROFILE_NAME,
        )
        extraction = evaluate_extraction(pred_artifact, gold_artifact, profile)
        result = {
            "layer_name": layer_name,
            "profile": PROFILE_NAME,
            "triples_path": str(triples_path),
            "evaluation_mode": "exact_neoolaf_evaluate_extraction",
            "extraction": extraction,
            "counts": {
                "pred_triples": len(triples),
                "pred_entities": sum(len(v) for v in pred_artifact.entities_by_doc.values()),
                "pred_relations": sum(len(v) for v in pred_artifact.relations_by_doc.values()),
                "gold_entities": sum(len(v) for v in gold_artifact.entities_by_doc.values()),
                "gold_relations": sum(len(v) for v in gold_artifact.relations_by_doc.values()),
            },
        }
        output_json.write_text(json.dumps(result, indent=2, ensure_ascii=False), encoding="utf-8")
        # Save a compact artifact summary for traceability.
        artifact_dump = {
            "method": pred_artifact.method,
            "dataset": pred_artifact.dataset,
            "profile": pred_artifact.profile,
            "run_id": pred_artifact.run_id,
            "metadata": pred_artifact.metadata,
            "documents": [getattr(d, "__dict__", str(d)) for d in pred_artifact.documents],
            "entities_by_doc_count": {k: len(v) for k, v in pred_artifact.entities_by_doc.items()},
            "relations_by_doc_count": {k: len(v) for k, v in pred_artifact.relations_by_doc.items()},
        }
        output_artifact.write_text(json.dumps(artifact_dump, indent=2, ensure_ascii=False), encoding="utf-8")
        triples_count = len(triples)

    summary = {
        "layer_name": layer_name,
        "triples_path": str(triples_path),
        "exact_eval_path": str(output_json),
        "evaluation_mode": "exact_neoolaf_evaluate_extraction",
        "pred_triples_loaded": triples_count,
    }
    summary.update(result.get("counts", {}))
    summary.update(summarize_extraction(extraction))
    summary_rows.append(summary)
    per_relation_rows.extend(extract_per_relation_rows(extraction, layer_name=layer_name))

summary_df = pd.DataFrame(summary_rows)
per_relation_df = pd.DataFrame(per_relation_rows)

summary_csv = EVAL_OUTPUT_ROOT / "summary_exact_eval.csv"
per_relation_csv = EVAL_OUTPUT_ROOT / "per_relation_exact_eval.csv"
summary_df.to_csv(summary_csv, index=False)
per_relation_df.to_csv(per_relation_csv, index=False)

print("Saved:", summary_csv)
print("Saved:", per_relation_csv)
display(summary_df)
display(per_relation_df.head(30))


In [ ]:

# ---------------------------------------------------------------------
# Native NeoOLAF final-export reference
# ---------------------------------------------------------------------
# This evaluates/loads the actual exported NeoOLAF Layer 12 output.
# It is NOT the same as the LLM-finalizer output from the layer12 checkpoint.

import subprocess
import sys

def find_metrics_summary(eval_dir: Path) -> Path | None:
    candidates = sorted(eval_dir.rglob("metrics.summary.json"))
    return candidates[0] if candidates else None


def run_native_export_gold_eval_if_needed() -> Path | None:
    if not NATIVE_EXPORT_DIR.exists():
        print("Native export directory not found; skipping native export reference:", NATIVE_EXPORT_DIR)
        return None

    existing = find_metrics_summary(NATIVE_EXPORT_EVAL_DIR)
    if existing is not None:
        print("Found existing native export metrics:", existing)
        return existing

    if not RUN_NATIVE_EXPORT_EVAL_IF_MISSING:
        print("Native export metrics not found and RUN_NATIVE_EXPORT_EVAL_IF_MISSING=False.")
        return None

    NATIVE_EXPORT_EVAL_DIR.mkdir(parents=True, exist_ok=True)

    cmd = [
        sys.executable, "-m", "neoolaf.evaluation", "evaluate",
        "--dataset", "xquality",
        "--method", "neoolaf",
        "--profile", NATIVE_EXPORT_PROFILE,
        "--input", str(NATIVE_EXPORT_DIR),
        "--gold", str(GOLD_PATH),
        "--output", str(NATIVE_EXPORT_EVAL_DIR),
    ]

    if SEED_ONTOLOGY.exists():
        cmd.extend(["--ontology-path", str(SEED_ONTOLOGY)])

    print("Running native final-export evaluation:")
    print(" ".join(cmd))

    result = subprocess.run(cmd, cwd=ROOT, text=True, capture_output=True)
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError("Native final-export evaluation failed")

    return find_metrics_summary(NATIVE_EXPORT_EVAL_DIR)


def extract_native_summary_row(summary_path: Path | None) -> dict[str, Any] | None:
    if summary_path is None or not Path(summary_path).exists():
        return None

    summary = json.loads(Path(summary_path).read_text(encoding="utf-8"))
    flat = flatten_scalars(summary)

    row = {
        "layer_name": "native_neoolaf_final_export",
        "evaluation_mode": "native_neoolaf_export_evaluate_cli",
        "profile": NATIVE_EXPORT_PROFILE,
        "metrics_summary_path": str(summary_path),
    }
    row.update(flat)

    # Use the same alias extraction as for LLM-finalizer rows.
    row["entity_precision"] = pick_metric(flat, [
        "entity.precision", "entities.precision", "entity_metrics.precision", "entity.p", "entity_p", "entities.p"
    ])
    row["entity_recall"] = pick_metric(flat, [
        "entity.recall", "entities.recall", "entity_metrics.recall", "entity.r", "entity_r", "entities.r"
    ])
    row["entity_f1"] = pick_metric(flat, [
        "entity.f1", "entities.f1", "entity_metrics.f1", "entity_f1", "entities.f1_score"
    ])
    row["relation_precision"] = pick_metric(flat, [
        "relation.precision", "relations.precision", "relation_metrics.precision", "relation.p", "relation_p", "relations.p"
    ])
    row["relation_recall"] = pick_metric(flat, [
        "relation.recall", "relations.recall", "relation_metrics.recall", "relation.r", "relation_r", "relations.r"
    ])
    row["relation_f1"] = pick_metric(flat, [
        "relation.f1", "relations.f1", "relation_metrics.f1", "relation_f1", "relations.f1_score"
    ])
    return row


native_metrics_summary_path = run_native_export_gold_eval_if_needed()
native_export_row = extract_native_summary_row(native_metrics_summary_path)

if native_export_row:
    print("Native NeoOLAF final export reference:")
    print({
        "profile": native_export_row.get("profile"),
        "entity_f1": native_export_row.get("entity_f1"),
        "relation_precision": native_export_row.get("relation_precision"),
        "relation_recall": native_export_row.get("relation_recall"),
        "relation_f1": native_export_row.get("relation_f1"),
    })
else:
    print("No native NeoOLAF final-export reference available.")



In [ ]:

# ---------------------------------------------------------------------
# Optional: evaluate native NeoOLAF layers with original evaluate_layer_state for side-by-side comparison
# ---------------------------------------------------------------------
from neoolaf.evaluation.runners.evaluate_layer_state import evaluate_run_layers

# Change this if your native run dir is different.
NATIVE_RUN_DIR_CANDIDATES = [
    RUNS_ROOT / "xquality_machine32" / "layer12_from_l11",  # sometimes only final layer exists here
    RUNS_ROOT / "xquality_machine32" / "full",
    RUNS_ROOT / "xquality_machine32",
]
NATIVE_RUN_DIR = next((p for p in NATIVE_RUN_DIR_CANDIDATES if p.exists()), None)
print("NATIVE_RUN_DIR:", NATIVE_RUN_DIR)

if NATIVE_RUN_DIR is not None:
    native_eval_dir = EVAL_OUTPUT_ROOT / "native_neoolaf_layer_evals"
    native_results = evaluate_run_layers(
        run_dir=NATIVE_RUN_DIR,
        gold_path=GOLD_PATH,
        profile_name=PROFILE_NAME,
        output_dir=native_eval_dir,
    )
    native_rows = []
    for r in native_results:
        extraction = r.get("extraction", {})
        row = {
            "layer_name": r.get("layer_name"),
            "state_path": r.get("state_path"),
            "profile": r.get("profile"),
            "evaluation_mode": "native_evaluate_layer_state",
        }
        row.update(r.get("counts", {}))
        row.update(summarize_extraction(extraction))
        native_rows.append(row)
    native_df = pd.DataFrame(native_rows)
    native_csv = EVAL_OUTPUT_ROOT / "native_neoolaf_summary_exact_eval.csv"
    native_df.to_csv(native_csv, index=False)
    print("Saved native comparison:", native_csv)
    display(native_df)
else:
    print("No native run directory found. Skipping native comparison.")


In [ ]:

# ---------------------------------------------------------------------
# Plot relation F1 by checkpoint + native NeoOLAF reference
# ---------------------------------------------------------------------
import matplotlib.pyplot as plt
import math

plot_df = summary_df.copy()

# Try to find a usable relation F1 column.
f1_col = None
for c in ["relation_f1", "relations.f1", "relation.f1", "relation_metrics.f1"]:
    if c in plot_df.columns and plot_df[c].notna().any():
        f1_col = c
        break

if f1_col:
    plt.figure(figsize=(13, 5))

    # LLM-finalizer checkpoint curve.
    plt.plot(
        plot_df["layer_name"],
        plot_df[f1_col],
        marker="o",
        label="LLM finalizer from checkpoint",
    )

    # Native final NeoOLAF reference.
    native_f1 = None
    if "native_export_row" in globals() and native_export_row:
        native_f1 = native_export_row.get("relation_f1")
        try:
            native_f1 = float(native_f1)
        except Exception:
            native_f1 = None

    if native_f1 is not None and not math.isnan(native_f1):
        plt.axhline(
            y=native_f1,
            linestyle="--",
            label=f"Native NeoOLAF final export F1={native_f1:.3f}",
        )

    plt.xticks(rotation=75, ha="right")
    plt.ylabel("Relation F1")
    plt.xlabel("Checkpoint / Layer")
    plt.title("LLM-finalizer checkpoint ablation vs native NeoOLAF final export")
    plt.legend()
    plt.tight_layout()

    fig_path = EVAL_OUTPUT_ROOT / "relation_f1_by_checkpoint_with_native_reference.png"
    plt.savefig(fig_path, dpi=180)
    plt.show()
    print("Saved plot:", fig_path)

    # Save a comparison table with the native reference appended.
    comparison_rows = []
    llm_comp = plot_df.copy()
    llm_comp["series"] = "llm_finalizer_from_checkpoint"
    comparison_rows.append(llm_comp)

    if "native_export_row" in globals() and native_export_row:
        native_df = pd.DataFrame([native_export_row])
        native_df["series"] = "native_neoolaf_final_export"
        comparison_rows.append(native_df)

    comparison_df = pd.concat(comparison_rows, ignore_index=True, sort=False)
    comparison_csv = EVAL_OUTPUT_ROOT / "comparison_with_native_reference.csv"
    comparison_df.to_csv(comparison_csv, index=False)
    print("Saved comparison table:", comparison_csv)
    display(comparison_df[["series", "layer_name", "evaluation_mode", "profile", "entity_f1", "relation_precision", "relation_recall", "relation_f1"]].tail(20))

else:
    print("Could not find a relation F1 column automatically. Available columns:")
    print(list(plot_df.columns))
